# Загружаем данные из API LSEG REFINITIV

In [1]:
import os
from datetime import datetime, timezone
import pandas as pd

import lseg.data as ld

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
FOLDER_PATH = r"C:\Courses\Innopolis\2026\ИИ Инженер\Data\Spark"

In [3]:
os.environ["NO_PROXY"] = "localhost"

In [4]:
ld.open_session()

<lseg.data.session.Definition object at 0x14227120ec0 {name='workspace'}>

In [5]:
# Список инструментов
instruments = [
    "AAPL.O",
    "MSFT.O",
    "NVDA.O"
]

# Получаем исторические данные
response = ld.get_history(
    universe=instruments,
    fields=[
        "TR.PriceClose",
        "TR.Volume"
    ],
    interval="daily",
    start="2026-01-01",
    end="2026-11-06"
)

In [6]:
response.head()

AAPL.O                MSFT.O                NVDA.O           
           Price Close    Volume Price Close    Volume Price Close     Volume
Date                                                                         
2026-01-02      271.01  37838054      472.94  25571567      188.85  148240462
2026-01-05      267.26  45647190      472.85  25250260      188.12  183529733
2026-01-06      262.36  52352090      478.51  23037665      187.24  176862604
2026-01-07      260.33  48309804      483.47  25564196      189.11  153543174
2026-01-08      259.04  50419337      478.11  18162616      185.04  172457048

In [7]:
# Закрываем сессию
ld.close_session()

# Bronze

In [17]:
df = response.swaplevel(axis=1).stack().reset_index()
df.columns = ["date", "instrument", "price_close", "volume"]

In [18]:
df.head()

,date,instrument,price_close,volume
0,2026-01-02,AAPL.O,271.01,37838054
1,2026-01-02,MSFT.O,472.94,25571567
2,2026-01-02,NVDA.O,188.85,148240462
3,2026-01-05,AAPL.O,267.26,45647190
4,2026-01-05,MSFT.O,472.85,25250260


In [19]:
# Предполагаем, что df содержит нормализованный результат получения данных из LSEG
bronze_df = df.copy()

# Метаданные загрузки
bronze_df["source"] = "LSEG_REFINITIV"
bronze_df["ingestion_time"] = pd.to_datetime(datetime.now(timezone.utc))
bronze_df["batch_id"] = "batch_001"

# Сохраняем исходный набор
bronze_df.to_csv(FOLDER_PATH + r"\bronze\lseg_prices.csv", index=False)

# Silver

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [4]:
spark = SparkSession.builder.appName("LSEG_Data_Architecture").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [5]:
# Считываем bronze слой из файла csv
bronze = spark.read.csv(FOLDER_PATH + r"\bronze\lseg_prices.csv", header=True, inferSchema=True)

In [6]:
bronze.printSchema()

root
 |-- date: string (nullable = true)
 |-- instrument: string (nullable = true)
 |-- price_close: double (nullable = true)
 |-- volume: integer (nullable = true)
 |-- source: string (nullable = true)
 |-- ingestion_time: string (nullable = true)
 |-- batch_id: string (nullable = true)



In [7]:
bronze.show(5)

+----------+----------+-----------+---------+--------------+--------------------+---------+
|      date|instrument|price_close|   volume|        source|      ingestion_time| batch_id|
+----------+----------+-----------+---------+--------------+--------------------+---------+
|2026-01-02|    AAPL.O|     271.01| 37838054|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|
|2026-01-02|    MSFT.O|     472.94| 25571567|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|
|2026-01-02|    NVDA.O|     188.85|148240462|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|
|2026-01-05|    AAPL.O|     267.26| 45647190|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|
|2026-01-05|    MSFT.O|     472.85| 25250260|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|
+----------+----------+-----------+---------+--------------+--------------------+---------+
only showing top 5 rows



In [8]:
# Вариант 1 (более контролируемый вариант, набор полей заранее известен)
silver = (
    bronze
    .select(
        F.to_date(F.col("date")).alias("date"),
        F.col("instrument").cast("string").alias("instrument"),
        F.col("price_close").cast("float").alias("price_close"),
        F.col("volume").cast("long").alias("volume"),
        F.col("source"),
        F.col("ingestion_time").cast("timestamp").alias("ingestion_time"),
        F.col("batch_id")
    )
)

silver.printSchema()

root
 |-- date: date (nullable = true)
 |-- instrument: string (nullable = true)
 |-- price_close: float (nullable = true)
 |-- volume: long (nullable = true)
 |-- source: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- batch_id: string (nullable = true)



In [9]:
silver.show(5)

+----------+----------+-----------+---------+--------------+--------------------+---------+
|      date|instrument|price_close|   volume|        source|      ingestion_time| batch_id|
+----------+----------+-----------+---------+--------------+--------------------+---------+
|2026-01-02|    AAPL.O|     271.01| 37838054|LSEG_REFINITIV|2026-11-09 21:10:...|batch_001|
|2026-01-02|    MSFT.O|     472.94| 25571567|LSEG_REFINITIV|2026-11-09 21:10:...|batch_001|
|2026-01-02|    NVDA.O|     188.85|148240462|LSEG_REFINITIV|2026-11-09 21:10:...|batch_001|
|2026-01-05|    AAPL.O|     267.26| 45647190|LSEG_REFINITIV|2026-11-09 21:10:...|batch_001|
|2026-01-05|    MSFT.O|     472.85| 25250260|LSEG_REFINITIV|2026-11-09 21:10:...|batch_001|
+----------+----------+-----------+---------+--------------+--------------------+---------+
only showing top 5 rows



In [10]:
# Вариант 2: перезаписать конкретные колонки (можно пропустить новую колонку из bronze)
silver = (
    bronze
    .withColumn("date", F.to_date(F.col("date")))
    .withColumn("instrument", F.col("instrument").cast("string"))
    .withColumn("price_close", F.col("price_close").cast("float"))
    .withColumn("volume", F.col("volume").cast("long"))
)

silver.printSchema()

root
 |-- date: date (nullable = true)
 |-- instrument: string (nullable = true)
 |-- price_close: float (nullable = true)
 |-- volume: long (nullable = true)
 |-- source: string (nullable = true)
 |-- ingestion_time: string (nullable = true)
 |-- batch_id: string (nullable = true)



In [11]:
# Очищаем от пропусков
silver = (
    silver
    .filter(
        (F.col("instrument").isNotNull()) &
        (F.col("date").isNotNull()) &
        (F.col("price_close").isNotNull()) &
        (F.col("volume") >= 0)
    )
)

In [12]:
# Убираем дубли
window = (
    Window
    .partitionBy("date", "instrument")
    .orderBy(F.col("ingestion_time").desc())
)

silver = (
    silver
    .withColumn("row_num", F.row_number().over(window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [13]:
silver.show(5)

+----------+----------+-----------+---------+--------------+--------------------+---------+
|      date|instrument|price_close|   volume|        source|      ingestion_time| batch_id|
+----------+----------+-----------+---------+--------------+--------------------+---------+
|2026-01-02|    AAPL.O|     271.01| 37838054|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|
|2026-01-02|    MSFT.O|     472.94| 25571567|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|
|2026-01-02|    NVDA.O|     188.85|148240462|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|
|2026-01-05|    AAPL.O|     267.26| 45647190|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|
|2026-01-05|    MSFT.O|     472.85| 25250260|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|
+----------+----------+-----------+---------+--------------+--------------------+---------+
only showing top 5 rows



In [14]:
silver.toPandas().to_csv(FOLDER_PATH + r"\silver\stock_prices.csv", index = False)

# Gold

In [15]:
from pyspark.sql import functions as F

In [16]:
# Считываем silver слой из файла csv
silver = spark.read.csv(FOLDER_PATH + r"\silver\stock_prices.csv", header=True, inferSchema=True)

window = (
    Window
    .partitionBy("instrument")
    .orderBy("date")
)

# Расчет предыдущей цены
gold = (
    silver
    .withColumn(
        "previous_price",
        F.lag("price_close").over(window)
    )
)

gold.show(5)

+----------+----------+-----------+--------+--------------+--------------------+---------+--------------+
|      date|instrument|price_close|  volume|        source|      ingestion_time| batch_id|previous_price|
+----------+----------+-----------+--------+--------------+--------------------+---------+--------------+
|2026-01-02|    AAPL.O|     271.01|37838054|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|          null|
|2026-01-05|    AAPL.O|     267.26|45647190|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|        271.01|
|2026-01-06|    AAPL.O|     262.36|52352090|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|        267.26|
|2026-01-07|    AAPL.O|     260.33|48309804|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|        262.36|
|2026-01-08|    AAPL.O|     259.04|50419337|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|        260.33|
+----------+----------+-----------+--------+--------------+--------------------+---------+--------------+
only showing top 5 rows



In [17]:
# Расчет дневной доходности
gold = (
    gold
    .withColumn("daily_return", F.col("price_close") / F.col("previous_price") - 1)
    .withColumn("price_change", F.col("price_close") - F.col("previous_price"))
)

gold.show(5)

+----------+----------+-----------+--------+--------------+--------------------+---------+--------------+--------------------+-------------------+
|      date|instrument|price_close|  volume|        source|      ingestion_time| batch_id|previous_price|        daily_return|       price_change|
+----------+----------+-----------+--------+--------------+--------------------+---------+--------------+--------------------+-------------------+
|2026-01-02|    AAPL.O|     271.01|37838054|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|          null|                null|               null|
|2026-01-05|    AAPL.O|     267.26|45647190|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|        271.01|-0.01383712778126267|              -3.75|
|2026-01-06|    AAPL.O|     262.36|52352090|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|        267.26|-0.01833420639078...| -4.899999999999977|
|2026-01-07|    AAPL.O|     260.33|48309804|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|        262.36|-0.0077374599

In [18]:
# Расчет средней активности торгов
volume_window = (
    Window
    .partitionBy("instrument")
    .orderBy("date")
    .rowsBetween(-4, 0)
)

gold = gold.withColumn("avg_volume_5d", F.avg("volume").over(volume_window))

gold.show(5)

+----------+----------+-----------+--------+--------------+--------------------+---------+--------------+--------------------+-------------------+--------------------+
|      date|instrument|price_close|  volume|        source|      ingestion_time| batch_id|previous_price|        daily_return|       price_change|       avg_volume_5d|
+----------+----------+-----------+--------+--------------+--------------------+---------+--------------+--------------------+-------------------+--------------------+
|2026-01-02|    AAPL.O|     271.01|37838054|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|          null|                null|               null|         3.7838054E7|
|2026-01-05|    AAPL.O|     267.26|45647190|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|        271.01|-0.01383712778126267|              -3.75|         4.1742622E7|
|2026-01-06|    AAPL.O|     262.36|52352090|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|        267.26|-0.01833420639078...| -4.899999999999977|4.527911133333

In [19]:
# Расчет кумулятивной доходности
gold = gold.withColumn(
    "first_price",
    F.first("price_close").over(
        window.rowsBetween(
            Window.unboundedPreceding,
            Window.unboundedFollowing
        )
    )
)

gold = gold.withColumn("total_return", F.col("price_close") / F.col("first_price") - 1)

gold.show(5)

+----------+----------+-----------+--------+--------------+--------------------+---------+--------------+--------------------+-------------------+--------------------+-----------+--------------------+
|      date|instrument|price_close|  volume|        source|      ingestion_time| batch_id|previous_price|        daily_return|       price_change|       avg_volume_5d|first_price|        total_return|
+----------+----------+-----------+--------+--------------+--------------------+---------+--------------+--------------------+-------------------+--------------------+-----------+--------------------+
|2026-01-02|    AAPL.O|     271.01|37838054|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|          null|                null|               null|         3.7838054E7|     271.01|                 0.0|
|2026-01-05|    AAPL.O|     267.26|45647190|LSEG_REFINITIV|2026-11-09 18:10:...|batch_001|        271.01|-0.01383712778126267|              -3.75|         4.1742622E7|     271.01|-0.01383712778126

In [20]:
gold = (
    gold
    .select(
        F.col("date"),
        F.col("instrument"),
        F.col("price_close"),
        F.col("volume"),
        F.round(F.col("daily_return") * 100, 2).alias("daily_return"),
        F.col("price_change"),
        F.col("avg_volume_5d"),
        F.round(F.col("total_return") * 100, 2).alias("total_return")
    )
)

In [21]:
gold.toPandas().to_csv(FOLDER_PATH + r"\gold\dm_prices.csv", index = False)